# Hybrid Engine — Integration Tests (v2)

**Doctor's deterministic routing + Claude Haiku LLM extraction**

### What changed from v1
The original harness fed answers as a *positional list* — answer[0] went to
question[0] regardless of which question the engine actually served.  
Because the engine's 7-phase schedule + skip/ask_if logic is dynamic, the
positions rarely matched, causing 2/9 headache checks to fail from harness
bugs, not engine bugs.

### Fix: keyword-matched answer dispatch
`answer_for_question()` inspects the live question's **`field`** name and
**`text`** against a per-scenario keyword map and returns the right scripted
answer, regardless of which turn it arrives in.  Any question not in the map
gets a safe default (`'No'` for booleans, `'not sure'` for free-text).

Run from `CAPSTONE2026_SPRING/` root.

In [1]:
import importlib, sys
sys.path.insert(0, '.')

import hybrid_engine
importlib.reload(hybrid_engine)
import hybrid_engine.extractor, hybrid_engine.session
importlib.reload(hybrid_engine.extractor)
importlib.reload(hybrid_engine.session)

from hybrid_engine import HybridIntakeSession

print('Available complaints:')
for c in HybridIntakeSession.available_complaints():
    print(f"  {c['complaint_id']:<30} {c['question_count']} questions  budget target={c['budget_target']}")

Available complaints:
  abdominal_pain                 59 questions  budget target=35
  back_pain                      51 questions  budget target=35
  chest_pain                     60 questions  budget target=35
  constipation                   31 questions  budget target=28
  cough                          53 questions  budget target=28
  dizziness                      39 questions  budget target=35
  dysuria                        38 questions  budget target=28
  fatigue                        40 questions  budget target=28
  fever                          61 questions  budget target=35
  headache                       55 questions  budget target=35
  hematuria                      49 questions  budget target=35
  joint_pain                     46 questions  budget target=28
  loose_stool                    53 questions  budget target=28
  loss_of_consciousness          62 questions  budget target=35
  oedema                         57 questions  budget target=35
  pain            

---
## Keyword-Matched Test Harness

`answer_for_question()` maps each live question to a scripted answer by
matching on the question's `field` name or keywords in its `text`.  
Unknown questions receive a safe neutral default so the session always
completes.

In [2]:
from pprint import pprint

all_results = []


def answer_for_question(question: dict, answer_map: dict, default_bool: str = 'No') -> str:
    """
    Return the scripted answer for *question* by matching its field name or
    text keywords against *answer_map*.

    answer_map keys are exact field names OR lowercase keyword substrings that
    appear in the question text.  Field-name matches take priority over
    text-keyword matches.  If nothing matches, returns default_bool for
    BOOLEAN questions and 'not sure' for everything else.
    """
    field = question.get('field', '')
    text  = question.get('text', '').lower()
    rtype = question.get('response_type', '')

    # 1. Exact field-name match (highest priority)
    if field in answer_map:
        return answer_map[field]

    # 2. Keyword-in-text match (any key that is a substring of the question text)
    for key, answer in answer_map.items():
        if key in text:
            return answer

    # 3. Safe default
    if 'BOOLEAN' in rtype:
        return default_bool
    return 'not sure'


def run_hybrid_test(complaint_id, patient_context, answer_map, label, checks,
                    default_bool='No'):
    print('=' * 70)
    print(f'TEST: {label}')
    print('=' * 70)

    session = HybridIntakeSession(
        complaint_id=complaint_id,
        patient_context=patient_context,
        verbose=False,
    )

    turn_count = 0
    while True:
        question = session.next_question()
        if question is None:
            break

        answer = answer_for_question(question, answer_map, default_bool)
        result = session.submit_answer(answer)
        turn_count += 1

        phase = question.get('phase', '')
        print(f"  [{phase}] {question['field']}")
        print(f"    Q  : {question['text'][:80]}")
        print(f"    Ans: {answer}")
        print(f"    ↳  : {result.canonical_answer!r}  (conf: {result.confidence})")
        if result.detail_value:
            print(f"    Det: {result.detail_value}")

    progress = session.get_progress()
    summary  = session.get_summary()

    print()
    print(f"Questions asked : {progress['questions_asked']} / budget {progress['budget_target']}")
    print(f"Escalation      : {progress['escalation_level']}")
    print(f"Red flags       : {[f['pattern'] for f in session.red_flags]}")
    print(f"Concepts covered: {progress['covered_concept_count']}")
    print()
    print('HPI:')
    print(summary['structured']['hpi'])
    print()

    passed = []
    print('CHECKS:')
    for check_label, condition in checks(session):
        status = '\033[92m PASS\033[0m' if condition else '\033[91m FAIL\033[0m'
        print(f'  {status}  {check_label}')
        passed.append(condition)

    print()
    return session, passed


print('Harness loaded.')

Harness loaded.


---
## 1. Headache — Thunderclap / SAH scenario

Classic subarachnoid haemorrhage presentation:
- Sudden-onset, maximal-at-onset, worst-ever headache  
- Neck stiffness, photophobia, confusion  
- Should fire thunderclap / SAH red flags → `urgent_escalation`

In [3]:
def headache_checks(session):
    s  = session.state
    sb = session.engine.state_bool
    return [
        ('worst_headache_of_life captured as True',  sb.get('worst_headache_of_life') is True),
        ('sudden_onset captured as True',            sb.get('sudden_onset') is True),
        ('maximal_at_onset captured as True',        sb.get('maximal_at_onset') is True),
        ('neck_stiffness captured as True',          sb.get('neck_stiffness') is True),
        ('confusion_or_ams captured as True',        sb.get('confusion_or_ams') is True),
        ('severity captured',                        s.get('severity') is not None),
        ('onset captured',                           s.get('onset') is not None),
        ('red flags triggered',                      len(session.red_flags) > 0),
        ('escalation above none',                    session.escalation_level != 'none'),
    ]

# Keys are exact field names OR lowercase substrings of the question text.
# Field-name matches take priority.
headache_answer_map = {
    # ── opening ──────────────────────────────────────────────────────
    'presenting_complaint_narrative':
        'I have a terrible headache that came on out of nowhere about two hours ago',

    # ── core_characterize ─────────────────────────────────────────────
    'onset':    'About two hours ago',
    'duration': 'Two hours and still going',
    'severity': '10 out of 10, worst pain of my life',

    # ── early_danger_screen ───────────────────────────────────────────
    'sudden_onset':               'Yes it came on like a thunderclap, instantly',
    'sudden_onset_details':       'Over a few seconds',
    'maximal_at_onset':           'Yes within the first minute it was already maximal',
    'maximal_at_onset_details':   'Peaked almost immediately',
    'worst_headache_of_life':     'Yes absolutely the worst headache I have ever had',
    'worst_headache_of_life_details': 'Far worse than any previous headache',
    'weakness':                   'No weakness',
    'speech_change':              'No trouble speaking',
    'worse_with_valsalva_or_waking': 'No',
    'transient_visual_obscurations': 'No',

    # ── extended_characterize ─────────────────────────────────────────
    'location':           'Back of the head and neck',
    'character':          'Throbbing and intense pressure',
    'timing':             'Constant has not changed at all',
    'course':             'Getting slightly worse if anything',
    'aggravating_factors':'Any movement makes it worse',
    'relieving_factors':  'Nothing makes it better',

    # ── critical_followup ─────────────────────────────────────────────
    'visual_loss':                    'No vision loss',
    'diplopia':                       'No double vision',
    'confusion_or_ams':               'Yes I am confused and not thinking straight',
    'confusion_or_ams_details':       'Constant since onset',
    'fever':                          'No fever',
    'neck_stiffness':                 'Yes my neck feels very stiff',
    'neck_stiffness_details':         'Cannot touch chin to chest',
    'recent_head_trauma':             'No head injury',
    'pregnancy_or_postpartum_context':'No I am not pregnant',
    'anticoagulant_use':              'No blood thinners',

    # ── high_priority_followup ────────────────────────────────────────
    'photophobia':            'Yes very sensitive to light',
    'nausea_or_vomiting':     'Yes nauseous and vomited once',
    'aura_status':            'No aura',
    'jaw_pain_with_chewing':  'No',
    'scalp_tenderness':       'No',
    'red_eye_or_eye_pain':    'No red eye',
    'new_or_progressive_pattern': 'Yes this is a completely new type of headache',
}

session1, passed1 = run_hybrid_test(
    complaint_id='headache',
    patient_context={'sex': 'female', 'age': 34},
    answer_map=headache_answer_map,
    label='HEADACHE — Thunderclap / SAH scenario',
    checks=headache_checks,
)
all_results.extend(passed1)

TEST: HEADACHE — Thunderclap / SAH scenario
  [opening] presenting_complaint_narrative
    Q  : Please tell me briefly what brought you in today and how this started.
    Ans: I have a terrible headache that came on out of nowhere about two hours ago
    ↳  : 'I have a terrible headache that came on out of nowhere about two hours ago'  (conf: high)
  [core_characterize] onset
    Q  : When did the headache start? If you can, tell me whether that was minutes, hours
    Ans: About two hours ago
    ↳  : '2 hours ago'  (conf: high)
  [core_characterize] duration
    Q  : How long has this headache been going on? If you can, tell me in minutes, hours,
    Ans: Two hours and still going
    ↳  : '2 hours'  (conf: high)
  [core_characterize] severity
    Q  : How severe is it on a scale of 0 to 10?
    Ans: 10 out of 10, worst pain of my life
    ↳  : '10'  (conf: high)
  [early_danger_screen] sudden_onset
    Q  : Did it come on suddenly rather than building up gradually?
    Ans: Yes it ca

---
## 2. Chest Pain — ACS scenario

Classic STEMI/ACS presentation:
- Central crushing chest pain radiating to jaw and left arm  
- Diaphoresis, shortness of breath, nausea  
- History of MI, hypertension, smoking  
- Should fire ACS red flags → `urgent_escalation` or higher

### Check note
The test asserts `sb.get('radiation') is True`.  
`radiation` is present in the question library (phase: `extended_characterize`)  
but the primary radiation question in the schedule is `arm_or_jaw_radiation`  
(phase: `early_danger_screen`).  Both fields are answered affirmatively so  
either field being True satisfies the spirit of the check.  
The check is written to cover both.

In [4]:
def chest_checks(session):
    s  = session.state
    sb = session.engine.state_bool
    # 'radiation' may be caught as arm_or_jaw_radiation (primary schedule question)
    # or as radiation (extended_characterize).  Accept either.
    radiation_positive = sb.get('radiation') is True or sb.get('arm_or_jaw_radiation') is True
    # 'nausea' is stored under 'nausea_or_vomiting'
    nausea_positive = sb.get('nausea_or_vomiting') is True or sb.get('nausea') is True
    return [
        ('severity captured',                 s.get('severity') is not None),
        ('radiation positive (arm_or_jaw or radiation field)',  radiation_positive),
        ('diaphoresis captured as True',      sb.get('diaphoresis') is True),
        ('shortness_of_breath as True',       sb.get('shortness_of_breath') is True),
        ('nausea captured as True',           nausea_positive),
        ('red flags triggered',               len(session.red_flags) > 0),
        ('escalation above none',             session.escalation_level != 'none'),
        ('onset captured',                    s.get('onset') is not None),
    ]

chest_answer_map = {
    # ── opening ──────────────────────────────────────────────────────
    'presenting_complaint_narrative':
        'Chest pain that started about 45 minutes ago while I was sitting watching TV',

    # ── core_characterize ─────────────────────────────────────────────
    'onset':    '45 minutes ago',
    'duration': '45 minutes, constant',
    'severity': '8 out of 10',
    'location': 'Centre of my chest',
    'character': 'Heavy and crushing like someone sitting on me',

    # ── early_danger_screen ───────────────────────────────────────────
    'rest_pain':              'Yes it is present at rest',
    'exertional_trigger':     'No I was just sitting down',
    'shortness_of_breath':    'Yes I am short of breath',
    'shortness_of_breath_details': 'At rest, moderately severe',
    'diaphoresis':            'Yes I am sweating a lot',
    'diaphoresis_details':    'Profuse sweating',
    'cocaine_or_stimulant_use': 'No',
    'arm_or_jaw_radiation':   'Yes it goes up into my jaw and down my left arm',
    'arm_or_jaw_radiation_details': 'Left arm and jaw',

    # ── extended_characterize ─────────────────────────────────────────
    'timing':             'Constant',
    'course':             'Getting worse',
    'radiation':          'Yes it goes to my left arm and jaw',   # belt-and-braces
    'radiation_details':  'Left arm and jaw',
    'aggravating_factors': 'Nothing makes it worse',
    'relieving_factors':  'Nothing makes it better',
    'functional_impact':  'Cannot do anything',

    # ── critical_followup ─────────────────────────────────────────────
    'nausea_or_vomiting':        'Yes I feel sick to my stomach',
    'syncope_or_near_syncope':   'No I have not fainted',
    'palpitations':              'No palpitations',
    'hemoptysis':                'No blood in cough',
    'tearing_to_back_quality':   'No it does not tear through to the back',

    # ── high_priority_followup ────────────────────────────────────────
    'associated_symptoms':       'Sweating and shortness of breath',
    'pleuritic_component':       'No',
    'positional_component':      'No',
    'cough':                     'No cough',
    'fever':                     'No fever',
    'leg_swelling':              'No leg swelling',
    'heartburn_or_reflux':       'No heartburn',
    'chest_wall_tenderness':     'No tenderness on pressing',
    'prior_cardiac_history':     'Yes I had a heart attack three years ago',
    'prior_cardiac_history_details': 'MI three years ago, stented',
    'clot_risk_history':         'No clot history',
    'family_history_early_heart_disease': 'No family history',
    'recent_immobility_or_travel': 'No',
}

session2, passed2 = run_hybrid_test(
    complaint_id='chest_pain',
    patient_context={'sex': 'male', 'age': 62},
    answer_map=chest_answer_map,
    label='CHEST PAIN — ACS scenario',
    checks=chest_checks,
)
all_results.extend(passed2)

TEST: CHEST PAIN — ACS scenario
  [opening] presenting_complaint_narrative
    Q  : Please tell me briefly what brought you in today and how this started.
    Ans: Chest pain that started about 45 minutes ago while I was sitting watching TV
    ↳  : 'Chest pain that started about 45 minutes ago while I was sitting watching TV'  (conf: high)
  [core_characterize] onset
    Q  : When did it start? If you can, tell me whether that was minutes, hours, days, we
    Ans: 45 minutes ago
    ↳  : '45 minutes ago'  (conf: high)
  [core_characterize] duration
    Q  : How should I understand the duration? Please answer in one of these formats: 'co
    Ans: 45 minutes, constant
    ↳  : 'constant for 45 minutes'  (conf: high)
  [core_characterize] severity
    Q  : How severe is it from 0 to 10?
    Ans: 8 out of 10
    ↳  : '8'  (conf: high)
  [core_characterize] location
    Q  : Where exactly do you feel it?
    Ans: Centre of my chest
    ↳  : 'Centre of chest'  (conf: high)
  [core_character

---
## 3. Seizure — Known epileptic, missed medication

Breakthrough seizure from non-compliance:
- Known epilepsy on levetiracetam, ran out 3 days ago  
- Generalised tonic-clonic, tongue biting, incontinence, postictal confusion  
- 90-second duration, witnessed

### Check corrections vs v1
| v1 check | Actual engine field | Fix |
|---|---|---|
| `sb.get('medication_compliance') is False` | No such field — compliance is embedded in `known_epilepsy` question text | Rewritten: check `sb.get('known_epilepsy') is True` (they're on AEDs) and `'miss' in s.get('known_epilepsy_details','').lower()` for missed doses |
| `sb.get('postictal_confusion') is True` | Field is `post_ictal_state` (SHORT_TEXT, not boolean) | Rewritten: check `s.get('post_ictal_state') is not None` |
| `s.get('severity') is not None` | Seizure has no severity question | Removed — replaced with `s.get('event_timing') is not None` |

In [5]:
def seizure_checks(session):
    s  = session.state
    sb = session.engine.state_bool
    # medication_compliance does not exist as a field — the engine captures
    # AED compliance inside known_epilepsy (bool) + known_epilepsy_details (text)
    on_aed      = sb.get('known_epilepsy') is True
    missed_dose = 'miss' in str(s.get('known_epilepsy_details', '')).lower()
    aed_missed  = on_aed and missed_dose
    return [
        ('event_timing captured',                        s.get('event_timing') is not None),
        ('prior_seizure_history captured as True',       sb.get('prior_seizure_history') is True),
        ('AED prescribed AND missed dose confirmed',     aed_missed),
        ('tongue_biting captured as True',               sb.get('tongue_biting') is True),
        ('post_ictal_state captured (postictal info)',   s.get('post_ictal_state') is not None),
        ('incontinence captured as True',                sb.get('incontinence') is True),
        ('duration captured',                            s.get('duration') is not None),
    ]

seizure_answer_map = {
    # ── opening ──────────────────────────────────────────────────────
    'presenting_complaint_narrative':
        'I had a seizure about two hours ago, my whole body was shaking',

    # ── core_characterize ─────────────────────────────────────────────
    'event_timing': 'About two hours ago',
    'duration':     'The shaking lasted about 90 seconds',
    'witnessed':    'My wife witnessed it',
    'prodrome':     'No I had no warning beforehand',
    'aura_status':  'No aura, it came without any warning',
    'seizure_type': 'Full body shaking, generalised tonic-clonic',

    # ── early_danger_screen ───────────────────────────────────────────
    'ongoing_confusion':        'No I am back to normal now',
    'focal_deficit_post_ictal': 'No weakness now',
    'fever_or_meningism':       'No fever or stiff neck',
    'head_injury':              'No head injury',

    # ── extended_characterize ─────────────────────────────────────────
    'aura_features':         'No aura features',
    'post_ictal_state':      'Yes I was confused for about 15 minutes after',
    'injury_sustained':      'No other injuries',
    'prior_seizure_history': 'Yes I have epilepsy, diagnosed five years ago',
    'prior_seizure_history_details': 'Epilepsy diagnosed 5 years ago, on levetiracetam',

    # ── critical_followup ─────────────────────────────────────────────
    'status_epilepticus_concern': 'No, it lasted 90 seconds and I recovered',
    'substance_or_overdose':      'No alcohol or drugs',
    'tongue_biting':              'Yes I bit my tongue on the side',
    'incontinence':               'Yes I lost bladder control',
    'hypoglycaemia_concern':      'No diabetes',

    # ── high_priority_followup ────────────────────────────────────────
    'associated_symptoms':  'Confusion after, tongue biting, incontinence',
    'triggered_or_provoked':'Yes I missed my medication for three days',
    'sleep_deprivation_trigger': 'I slept normally last night',
    'alcohol_withdrawal':   'No I do not drink heavily',
    'new_medication_or_toxin': 'No new medications',
    'known_epilepsy':       'Yes I am on levetiracetam 500mg twice daily',
    # Detail captures the missed-dose fact — this is what the check inspects
    'known_epilepsy_details': 'Levetiracetam 500mg BD — I ran out three days ago and missed my doses',
    'family_history_epilepsy': 'No family history of epilepsy',
    'prior_brain_pathology': 'No brain tumour or stroke',
}

session3, passed3 = run_hybrid_test(
    complaint_id='seizure',
    patient_context={'sex': 'male', 'age': 32},
    answer_map=seizure_answer_map,
    label='SEIZURE — Known epileptic, missed medication',
    checks=seizure_checks,
)
all_results.extend(passed3)

TEST: SEIZURE — Known epileptic, missed medication
  [opening] presenting_complaint_narrative
    Q  : Please tell me briefly what happened and what brought you in today.
    Ans: I had a seizure about two hours ago, my whole body was shaking
    ↳  : 'I had a seizure about two hours ago, my whole body was shaking'  (conf: high)
  [core_characterize] event_timing
    Q  : When did the episode happen?
    Ans: About two hours ago
    ↳  : '2 hours ago'  (conf: high)
  [core_characterize] duration
    Q  : How long did it last? If you can, tell me in minutes, hours, days, weeks, months
    Ans: The shaking lasted about 90 seconds
    ↳  : '90 seconds'  (conf: high)
  [core_characterize] witnessed
    Q  : Did anyone see it happen?
    Ans: My wife witnessed it
    ↳  : 'Wife witnessed the event'  (conf: high)
  [core_characterize] prodrome
    Q  : Did you feel anything just before it, such as light-headedness, a rising feeling
    Ans: No I had no warning beforehand
    ↳  : 'no'  (conf

---
## 4. Palpitations — Exertional with presyncope

Exertional palpitations with near-syncope:
- Racing heart during exercise, felt faint  
- Diaphoresis (heavy sweating)  
- No chest pain, no shortness of breath

### Check corrections vs v1
| v1 check | Actual engine field | Fix |
|---|---|---|
| `sb.get('syncope') is True` or `sb.get('near_syncope') is True` | Field is `syncope_or_near_syncope` | Check updated to use correct field |
| `sb.get('diaphoresis') is True` | `diaphoresis` does not exist in palpitations schema | Rewritten: check that `'sweat'` appears in `associated_symptoms` or `triggers` text |
| `sb.get('chest_pain') is False` | Field is `chest_pain` ✓ | Unchanged |
| `sb.get('shortness_of_breath') is False` | Field is `shortness_of_breath` ✓ | Unchanged |

In [6]:
def palp_checks(session):
    s  = session.state
    sb = session.engine.state_bool
    # diaphoresis is not a palpitations schema field; sweating appears in associated_symptoms or triggers
    sweat_mentioned = any(
        'sweat' in str(s.get(f, '')).lower()
        for f in ('associated_symptoms', 'triggers', 'functional_impact')
    )
    return [
        ('onset captured',                           s.get('onset') is not None),
        ('severity captured',                        s.get('severity') is not None),
        ('syncope_or_near_syncope captured as True', sb.get('syncope_or_near_syncope') is True),
        ('sweating mentioned in session state',      sweat_mentioned),
        ('chest_pain captured as False',             sb.get('chest_pain') is False),
        ('shortness_of_breath captured as False',    sb.get('shortness_of_breath') is False),
    ]

palp_answer_map = {
    # ── opening ──────────────────────────────────────────────────────
    'presenting_complaint_narrative':
        'My heart was racing during my run this afternoon and I nearly fainted',

    # ── core_characterize ─────────────────────────────────────────────
    'onset':     'This afternoon during my run',
    'duration':  'About 5 minutes',
    'frequency': 'Just this one episode today',
    'character': 'Racing and pounding',
    'severity':  '7 out of 10, very distressing',

    # ── early_danger_screen ───────────────────────────────────────────
    'syncope_or_near_syncope':         'Yes I nearly fainted during the episode',
    'syncope_or_near_syncope_details': 'Near-faint, no loss of consciousness',
    'chest_pain':                      'No chest pain',
    'shortness_of_breath':             'No trouble breathing',
    'sudden_onset':                    'Yes starts suddenly like a switch',

    # ── extended_characterize ─────────────────────────────────────────
    'triggers':          'Exercise brings it on',
    'functional_impact': 'Had to stop running',
    'anxiety_context':   'No anxiety context, only during exercise',

    # ── critical_followup ─────────────────────────────────────────────
    'exertional_trigger':              'Yes it came on during my run',
    'exertional_trigger_details':      'Running, within a few minutes of starting',
    'rapid_or_irregular_rhythm':       'Very fast, not sure if irregular',
    'family_history_sudden_death':     'No family history of sudden death',
    'structural_heart_disease':        'No known heart condition',

    # ── high_priority_followup ────────────────────────────────────────
    # Sweating captured here — this is what the check inspects
    'associated_symptoms':     'Dizziness and I was sweating heavily during the episode',
    'positional_or_vagal_trigger': 'No positional trigger',
    'caffeine_alcohol_trigger':    'No caffeine or alcohol link',
    'thyroid_symptoms':            'No thyroid symptoms',
    'new_medication':              'No medications',
    'prior_episodes':              'No prior episodes',
    'electrolyte_symptoms':        'No muscle cramps',
}

session4, passed4 = run_hybrid_test(
    complaint_id='palpitations',
    patient_context={'sex': 'female', 'age': 28},
    answer_map=palp_answer_map,
    label='PALPITATIONS — Exertional with presyncope',
    checks=palp_checks,
)
all_results.extend(passed4)

TEST: PALPITATIONS — Exertional with presyncope
  [opening] presenting_complaint_narrative
    Q  : Please tell me briefly what brought you in today and how this started.
    Ans: My heart was racing during my run this afternoon and I nearly fainted
    ↳  : 'My heart was racing during my run this afternoon and I nearly fainted'  (conf: high)
  [core_characterize] onset
    Q  : When did you first notice them? If you can, tell me whether that was minutes, ho
    Ans: This afternoon during my run
    ↳  : 'this afternoon'  (conf: high)
  [core_characterize] duration
    Q  : How long does each episode last—seconds, minutes, or longer?
    Ans: About 5 minutes
    ↳  : '5 minutes'  (conf: high)
  [core_characterize] frequency
    Q  : How often are they happening?
    Ans: Just this one episode today
    ↳  : '1'  (conf: high)
  [core_characterize] character
    Q  : How would you describe the feeling—fast, irregular, pounding, fluttering, or a s
    Ans: Racing and pounding
    ↳  : 'Ra

---
## Summary

In [7]:
total  = len(all_results)
passed = sum(all_results)
failed = total - passed

print(f'\n{"=" * 40}')
print(f'  Results  : {passed}/{total} passed')
if failed:
    print(f'  FAILED   : {failed}')
    print()
    print('  Failing checks:')
    labels = (
        # headache (9)
        'worst_headache_of_life True','sudden_onset True','maximal_at_onset True',
        'neck_stiffness True','confusion_or_ams True','severity captured',
        'onset captured','red flags triggered','escalation above none',
        # chest_pain (8)
        'severity captured','radiation positive','diaphoresis True',
        'shortness_of_breath True','nausea True','red flags triggered',
        'escalation above none','onset captured',
        # seizure (7)
        'event_timing captured','prior_seizure_history True','AED missed',
        'tongue_biting True','post_ictal_state captured','incontinence True',
        'duration captured',
        # palpitations (6)
        'onset captured','severity captured','syncope_or_near_syncope True',
        'sweating mentioned','chest_pain False','shortness_of_breath False',
    )
    for label, result in zip(labels, all_results):
        if not result:
            print(f'    FAIL  {label}')
else:
    print('  All checks passed! ✓')
print(f'{"=" * 40}')


  Results  : 30/30 passed
  All checks passed! ✓
